In [49]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, root_mean_squared_error , mean_absolute_error , mean_squared_error
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay , roc_curve, RocCurveDisplay, roc_auc_score


from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor 


from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import   GradientBoostingRegressor, StackingRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb

from sklearn.preprocessing import OneHotEncoder , StandardScaler, LabelEncoder
from sklearn.compose import make_column_transformer , make_column_selector
 

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.pipeline import Pipeline



import warnings
warnings.filterwarnings("ignore")

os.chdir(r"E:\PML\Datasets\Cases\Vehicle Silhouettes")

In [50]:
vehicle = pd.read_csv('Vehicle.csv')

X,y = vehicle.drop('Class', axis = 1) , vehicle['Class']

enc = LabelEncoder()
le_y = enc.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X,le_y, test_size = 0.3, random_state=26 ,stratify=le_y)


In [51]:


vehicle['Class'].unique()

<ArrowStringArray>
['van', 'saab', 'bus', 'opel']
Length: 4, dtype: str

In [56]:
gb = GradientBoostingClassifier(random_state=26)
gb.fit(X_train, y_train)
y_pred_prob = gb.predict_proba(X_test)
y_pred_prob

array([[1.43558203e-04, 2.63658671e-02, 9.72686140e-01, 8.04434367e-04],
       [6.00096198e-01, 2.21345968e-02, 1.72180525e-02, 3.60551153e-01],
       [9.97332823e-01, 1.33234575e-03, 9.02216856e-04, 4.32614007e-04],
       ...,
       [9.98412988e-01, 9.98005589e-04, 5.57075562e-04, 3.19307378e-05],
       [5.18035301e-03, 7.23266341e-01, 2.59397502e-01, 1.21558040e-02],
       [3.85205539e-05, 1.39448231e-03, 1.76511503e-02, 9.80915847e-01]],
      shape=(254, 4))

In [53]:
print("AUC  (OVO)", roc_auc_score(y_test, y_pred_prob , multi_class='ovo'))
print("AUC  (OVO)", roc_auc_score(y_test, y_pred_prob , multi_class='ovr'))

AUC  (OVO) 0.9305661879520052
AUC  (OVO) 0.9294567746223928


In [54]:
# 0 to 1  -> log vals always -ve

In [55]:
log_loss(y_test, y_pred_prob)

0.5045919125374988

In [57]:
 
y_pred_prob.shape

(254, 4)

In [ ]:

learning_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimator = [25,50,100] 
scores = []
for d in tqdm(max_depths):
      for e in n_estimator:
          for r in learning_rates:
                gbr = GradientBoostingClassifier(random_state=26,n_estimators=e,max_depth=d,learning_rate=r)
                gbr.fit(X_train,y_train)
                y_pred_proba = gbr.predict_proba(X_test)
                scores.append([d,e,r, log_loss(y_test, y_pred_proba)])

              
df_scores = pd.DataFrame(scores, columns=['depth','estimator','learning_rate','log_loss'])
df_scores.sort_values('log_loss')

In [66]:

learning_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimator = [25,50,100] 
scores = []
for d in tqdm(max_depths):
      for e in n_estimator:
          for r in learning_rates:
                gbr = GradientBoostingClassifier(random_state=26,n_estimators=e,max_depth=d,learning_rate=r)
                gbr.fit(X_train,y_train)
                y_pred_proba = gbr.predict_proba(X_test)
                scores.append([d,e,r, -log_loss(y_test, y_pred_proba)])

              
df_scores = pd.DataFrame(scores, columns=['depth','estimator','learning_rate','log_loss'])


100%|██████████| 4/4 [01:33<00:00, 23.25s/it]


In [67]:
df_scores.sort_values('log_loss')


,depth,estimator,learning_rate,log_loss
7,NaN,25,0.702222,-2.891586
17,NaN,50,0.702222,-2.891586
27,NaN,100,0.702222,-2.891586
9,NaN,25,0.900000,-2.701495
19,NaN,50,0.900000,-2.701495
...,...,...,...,...
43,2.0,50,0.306667,-0.495539
34,2.0,25,0.405556,-0.490844
51,2.0,100,0.108889,-0.489373
44,2.0,50,0.405556,-0.480414
